In [1]:
# W4D3: Model Serialisation ? joblib & pickle
# One reproducible workflow: train, save with both formats, reload, verify, and record output evidence.
from pathlib import Path
import json
import hashlib
import pickle
import joblib
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

SEED = 42
OUT = Path("../outputs/w4d3_serialisation")
OUT.mkdir(parents=True, exist_ok=True)
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                    stratify=y, random_state=SEED)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=SEED))
model.fit(X_train, y_train)
# Save the fitted pipeline so the preprocessing and estimator travel together.
joblib_path, pickle_path = OUT / "breast_cancer_pipeline.joblib", OUT / "breast_cancer_pipeline.pkl"
joblib.dump(model, joblib_path, compress=3)
with pickle_path.open("wb") as handle:
    pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
# Load each artifact and assert that both produce the exact same probabilities as the original.
loaded_joblib = joblib.load(joblib_path)
with pickle_path.open("rb") as handle:
    loaded_pickle = pickle.load(handle)
expected = model.predict_proba(X_test)
for label, restored in (("joblib", loaded_joblib), ("pickle", loaded_pickle)):
    probabilities = restored.predict_proba(X_test)
    assert np.array_equal(expected, probabilities), f"{label} round-trip changed predictions"
    assert np.array_equal(model.predict(X_test), restored.predict(X_test)), f"{label} round-trip changed classes"
    print(f"{label} reload: prediction parity verified")
pred = loaded_joblib.predict(X_test)
prob = loaded_joblib.predict_proba(X_test)[:, 1]
metrics = {"accuracy": accuracy_score(y_test, pred), "precision": precision_score(y_test, pred),
           "recall": recall_score(y_test, pred), "roc_auc": roc_auc_score(y_test, prob),
           "confusion_matrix": confusion_matrix(y_test, pred).tolist()}
# Record hashes, file sizes, dependency versions, and quality metrics as portable evidence.
import sklearn
manifest = {"model_type": "StandardScaler + LogisticRegression", "random_seed": SEED,
            "python_note": "Load pickle/joblib artifacts only from trusted sources.",
            "versions": {"scikit_learn": sklearn.__version__, "joblib": joblib.__version__},
            "artifacts": {p.name: {"bytes": p.stat().st_size,
                                    "sha256": hashlib.sha256(p.read_bytes()).hexdigest()}
                          for p in (joblib_path, pickle_path)},
            "metrics": metrics}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
pd.DataFrame([{"format": "joblib", "path": str(joblib_path), "bytes": joblib_path.stat().st_size},
              {"format": "pickle", "path": str(pickle_path), "bytes": pickle_path.stat().st_size}]).to_csv(OUT / "artifact_sizes.csv", index=False)
print("Reloaded model metrics:", json.dumps(metrics, indent=2))
print("Saved artifacts:\n", pd.read_csv(OUT / "artifact_sizes.csv").to_string(index=False))
print("Self-review: reproducible split, pipeline saved, both formats reloaded, prediction parity asserted, manifest recorded.")
print("Evidence:", OUT.resolve())


joblib reload: prediction parity verified
pickle reload: prediction parity verified
Reloaded model metrics: {
  "accuracy": 0.986013986013986,
  "precision": 0.9888888888888889,
  "recall": 0.9888888888888889,
  "roc_auc": 0.9976939203354298,
  "confusion_matrix": [
    [
      52,
      1
    ],
    [
      1,
      89
    ]
  ]
}
Saved artifacts:
 format                                                        path  bytes
joblib ..\outputs\w4d3_serialisation\breast_cancer_pipeline.joblib   1814
pickle    ..\outputs\w4d3_serialisation\breast_cancer_pipeline.pkl   2026
Self-review: reproducible split, pipeline saved, both formats reloaded, prediction parity asserted, manifest recorded.
Evidence: C:\cynarisis-internship\outputs\w4d3_serialisation
